# EDA 05: Regional Demand Patterns, Category Shares & Weather Correlation

This notebook evaluates product category demand variations, demand volatility (Coefficient of Variation), and correlation with synthetic weather variables.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

df_sales = pd.read_parquet('data/processed/sales_fact.parquet')
df_prod = pd.read_parquet('data/processed/product_dim.parquet')
df_weather = pd.read_parquet('data/processed/weather_dim.parquet')

df_merged = df_sales.merge(df_prod[['product_id', 'category', 'sub_category']], on='product_id', how='left')
print(f"Sales with Category Info: {len(df_merged):,}")


Sales with Category Info: 1,143,942


In [2]:
# Regional Category Sales Heatmap
cat_region_pivot = pd.pivot_table(
    df_merged,
    values='total_sales',
    index='category',
    columns='dataset_source',
    aggfunc='sum',
    fill_value=0
)

plt.figure(figsize=(10, 8))
sns.heatmap(cat_region_pivot / 1e6, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Category Revenue Breakdown by Dataset Source ($ Millions)')
plt.xlabel('Dataset Source')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()


In [3]:
# Demand Volatility Analysis (Coefficient of Variation = std / mean)
daily_reg_sales = df_sales.groupby(['region', 'date'])['total_sales'].sum().reset_index()

cv_summary = daily_reg_sales.groupby('region')['total_sales'].agg(
    mean_sales='mean',
    std_sales='std',
    days_recorded='count'
).reset_index()

cv_summary['cv'] = cv_summary['std_sales'] / cv_summary['mean_sales']
cv_summary['volatility_tier'] = pd.qcut(cv_summary['cv'], q=3, labels=['Low Volatility', 'Medium Volatility', 'High Volatility'])

print(cv_summary.sort_values(by='cv', ascending=False).head(10))

plt.figure(figsize=(10, 5))
sns.barplot(data=cv_summary.sort_values(by='cv', ascending=False).head(10), x='cv', y='region', palette='OrRd_r')
plt.title('Top 10 Regions by Demand Volatility (Coefficient of Variation)')
plt.xlabel('CV (std / mean)')
plt.ylabel('Region')
plt.show()


   region  mean_sales   std_sales  days_recorded        cv  volatility_tier
32     RS  182.488012  237.023514           1987  1.298844  High Volatility
35     SP  120.585336  147.179191          69835  1.220540  High Volatility
1      AM  372.666667  448.023809              3  1.202211  High Volatility
18     MG  125.774403  145.141457           7920  1.153982  High Volatility
30     RN  176.149020  203.062907             51  1.152790  High Volatility
16     GO  127.819568  143.788326            463  1.124932  High Volatility
11     ES  142.302233  158.499770            318  1.113825  High Volatility
10     DF  118.235995  130.818062            824  1.106415  High Volatility
33     SC  164.085572  181.453308           3663  1.105846  High Volatility
28     PR  157.531837  171.101711           7665  1.086141  High Volatility


In [4]:
# Weather Parameter Correlation Analysis
df_sales['date_clean'] = df_sales['date'].dt.date
df_weather['date_clean'] = df_weather['date'].dt.date

daily_sales_reg = df_sales.groupby(['region', 'date_clean'])['total_sales'].sum().reset_index()
weather_merged = daily_sales_reg.merge(df_weather, on=['region', 'date_clean'], how='inner')

corr_matrix = weather_merged[['total_sales', 'temperature_c', 'precipitation_mm', 'humidity_pct']].corr()
print("Correlation Matrix (Daily Sales vs Synthetic Weather):")
print(corr_matrix)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation: Daily Sales vs Synthetic Weather Parameters')
plt.show()


Correlation Matrix (Daily Sales vs Synthetic Weather):
                  total_sales  temperature_c  precipitation_mm  humidity_pct
total_sales          1.000000       0.013951          0.010732     -0.005184
temperature_c        0.013951       1.000000         -0.000388      0.002901
precipitation_mm     0.010732      -0.000388          1.000000      0.284234
humidity_pct        -0.005184       0.002901          0.284234      1.000000
